<a href="https://colab.research.google.com/github/Yi-LingT/Analytics-Projects/blob/main/UC_DAVIS_SQL_for_Data_Science_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install duckdb==1.1.1 pandas==2.2.2 pyarrow==17.0.0

import os, pandas as pd, numpy as np
print("Files in /content:", os.listdir("/content"))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 17.9 MB/s eta 0:00:00
Files in /content: ['.config', 'honey_production.csv', 'state_lookup.csv', 'cheese_production.csv', 'coffee_production.csv', 'egg_production.csv', 'yogurt_production.csv', 'milk_production.csv', 'sample_data']


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content")
FILES = {
    "state_lookup": "state_lookup.csv",
    "egg_production": "egg_production.csv",
    "coffee_production": "coffee_production.csv",
    "milk_production": "milk_production.csv",
    "honey_production": "honey_production.csv",
    "cheese_production": "cheese_production.csv",
    "yogurt_production": "yogurt_production.csv",
}

def load_csv_safe(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, low_memory=False, encoding="utf-8")

def clean_numeric_columns(df: pd.DataFrame, prefer_int=True) -> pd.DataFrame:
    cat_like = {"state", "period", "geo_level", "domain"}
    out = df.copy()
    for col in out.columns:
        cl = col.strip().lower()
        if cl in cat_like:
            continue
        if out[col].dtype == object:
            tmp = pd.to_numeric(
                out[col].astype(str).str.replace(",", "").str.strip(),
                errors="coerce"
            )
            # 若能成功轉出多數值，就採用
            if tmp.notna().mean() > 0.5:
                if prefer_int and tmp.dropna().mod(1).eq(0).all():
                    out[col] = tmp.astype("Int64")
                else:
                    out[col] = tmp.astype(float)
    return out

tables = {}
for tname, fname in FILES.items():
    df = load_csv_safe(DATA_DIR / fname)
    df = clean_numeric_columns(df)
    tables[tname] = df

{t: df.head(3) for t, df in tables.items()}


{'state_lookup':      State  State_ANSI
 0  ALABAMA           1
 1   ALASKA           2
 2  ARIZONA           4,
 'egg_production':    Year Period Geo_Level  State_ANSI  Commodity_ID  Value
 0  2011    APR     STATE        15.0             7   <NA>
 1  2011    AUG     STATE        15.0             7   <NA>
 2  2011    FEB     STATE        15.0             7   <NA>,
 'coffee_production':    Year Period Geo_Level  State_ANSI  Commodity_ID     Value
 0  1957   YEAR     STATE          15             1  18496000
 1  1962   YEAR     STATE          15             1  13392000
 2  1960   YEAR     STATE          15             1  13272000,
 'milk_production':    Year Period Geo_Level  State_ANSI  Commodity_ID Domain       Value
 0  2023    APR     STATE         4.0             5  TOTAL   428000000
 1  2023    APR     STATE         6.0             5  TOTAL  3543000000
 2  2023    APR     STATE         8.0             5  TOTAL   444000000,
 'honey_production':    Year Geo_Level  State_ANSI  Commod

In [ ]:
import duckdb
con = duckdb.connect(database=":memory:")

# 註冊為表（也可改用 CREATE TABLE AS SELECT）
for tname, df in tables.items():
    con.register(tname, df)

print("Registered tables:", list(tables.keys()))

# 範例 SQL：查 5 筆
con.execute("""
SELECT Year, Period, Geo_Level, State_ANSI, Commodity_ID, Value
FROM egg_production
LIMIT 5
""").df()


Registered tables: ['state_lookup', 'egg_production', 'coffee_production', 'milk_production', 'honey_production', 'cheese_production', 'yogurt_production']


,Year,Period,Geo_Level,State_ANSI,Commodity_ID,Value
0,2011,APR,STATE,15.0,7,NaN
1,2011,AUG,STATE,15.0,7,NaN
2,2011,FEB,STATE,15.0,7,NaN
3,2011,JAN,STATE,15.0,7,NaN
4,2011,JUL,STATE,15.0,7,NaN


In [ ]:
import duckdb
con = duckdb.connect(database=":memory:")

# 註冊為表（也可改用 CREATE TABLE AS SELECT）
for tname, df in tables.items():
    con.register(tname, df)

print("Registered tables:", list(tables.keys()))

# 範例 SQL：查 5 筆
con.execute("""
SELECT Year, Period, Geo_Level, State_ANSI, Commodity_ID, Value
FROM egg_production
LIMIT 5
""").df()


Registered tables: ['state_lookup', 'egg_production', 'coffee_production', 'milk_production', 'honey_production', 'cheese_production', 'yogurt_production']


,Year,Period,Geo_Level,State_ANSI,Commodity_ID,Value
0,2011,APR,STATE,15.0,7,NaN
1,2011,AUG,STATE,15.0,7,NaN
2,2011,FEB,STATE,15.0,7,NaN
3,2011,JAN,STATE,15.0,7,NaN
4,2011,JUL,STATE,15.0,7,NaN


In [ ]:
def is_numeric_dtype_duckdb(con, table, column):
    info = con.execute(f"DESCRIBE {table}").df()
    row = info[info['column_name'].str.lower() == column.lower()]
    if row.empty:
        return False
    dtype = row['column_type'].iloc[0].lower()
    return any(k in dtype for k in ['int', 'hugeint', 'decimal', 'double', 'float', 'real'])

def dq_report_for_table(con, table):
    cols = con.execute(f"DESCRIBE {table}").df()['column_name'].tolist()
    n = con.execute(f"SELECT COUNT(*) as n FROM {table}").df()['n'].iloc[0]
    rows = []
    for col in cols:
        miss = con.execute(f"SELECT SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS m FROM {table}").df()['m'].iloc[0]
        miss_pct = (miss / n * 100) if n else 0.0

        if is_numeric_dtype_duckdb(con, table, col):
            q = con.execute(f"""
                SELECT
                  quantile_cont({col}, 0.25) AS q1,
                  quantile_cont({col}, 0.75) AS q3
                FROM {table}
                WHERE {col} IS NOT NULL
            """).df()
            q1, q3 = q['q1'].iloc[0], q['q3'].iloc[0]
            num_outliers, examples = 0, []
            if q1 is not None and q3 is not None:
                iqr = q3 - q1
                lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
                num_outliers = con.execute(f"""
                    SELECT COUNT(*) AS c
                    FROM {table}
                    WHERE {col} IS NOT NULL AND ({col} < {lower} OR {col} > {upper})
                """).df()['c'].iloc[0]
                examples = con.execute(f"""
                    SELECT {col} AS val
                    FROM {table}
                    WHERE {col} IS NOT NULL AND ({col} < {lower} OR {col} > {upper})
                    LIMIT 5
                """).df()['val'].tolist()
            rows.append({
                "table": table, "column": col, "is_numeric": True,
                "missing_%": round(miss_pct,2), "suspicious_%": 0.00,
                "num_outliers": int(num_outliers), "outlier_examples": examples
            })
        else:
            rows.append({
                "table": table, "column": col, "is_numeric": False,
                "missing_%": round(miss_pct,2), "suspicious_%": 0.00,
                "num_outliers": 0, "outlier_examples": []
            })
    return pd.DataFrame(rows)

reports = [dq_report_for_table(con, t) for t in tables.keys()]
dq_all = pd.concat(reports, ignore_index=True).sort_values(["table","column"]).reset_index(drop=True)
dq_all.head(20)


,table,column,is_numeric,missing_%,suspicious_%,num_outliers,outlier_examples
0,cheese_production,Commodity_ID,True,0.00,0.0,0,[]
1,cheese_production,Domain,False,0.00,0.0,0,[]
2,cheese_production,Geo_Level,False,0.00,0.0,0,[]
3,cheese_production,Period,False,0.00,0.0,0,[]
4,cheese_production,State_ANSI,True,0.01,0.0,0,[]
5,cheese_production,Value,True,0.00,0.0,4169,"[3543000000, 1408000000, 1342000000, 141300000..."
6,cheese_production,Year,True,0.00,0.0,0,[]
7,coffee_production,Commodity_ID,True,0.00,0.0,0,[]
8,coffee_production,Geo_Level,False,0.00,0.0,0,[]
9,coffee_production,Period,False,0.00,0.0,0,[]


In [ ]:
dq_path = "/content/dq_report.csv"
dq_all.to_csv(dq_path, index=False)
dq_path


'/content/dq_report.csv'

In [ ]:
def suspicious_rate_for_text_as_number(con, table, column):
    res = con.execute(f"""
        WITH base AS (
          SELECT {column} AS c FROM {table} WHERE {column} IS NOT NULL
        ),
        cand AS (
          SELECT c FROM base WHERE c ~ '^[0-9 ,.-]+$'
        ),
        ok AS (
          SELECT c FROM cand
          WHERE TRY_CAST(REPLACE(REPLACE(TRIM(c), ',', ''), ' ', '') AS DOUBLE) IS NOT NULL
        )
        SELECT
          (SELECT COUNT(*) FROM base) AS n_all,
          (SELECT COUNT(*) FROM cand) AS n_cand,
          (SELECT COUNT(*) FROM cand) - (SELECT COUNT(*) FROM ok) AS n_bad
    """).df()
    if res.empty or res['n_all'].iloc[0] == 0:
        return 0.0
    n_all, n_cand, n_bad = res.loc[0, ['n_all','n_cand','n_bad']]
    return round((n_bad / n_all) * 100, 2)

for t in tables.keys():
    info = con.execute(f"DESCRIBE {t}").df()
    for col, dtype in zip(info['column_name'], info['column_type']):
        if 'varchar' in dtype.lower():
            sus = suspicious_rate_for_text_as_number(con, t, col)
            if sus > 0:
                print(f"[{t}.{col}] suspicious% ≈ {sus}")


Scenario:

Data Scientist at USDA (United States Department of Agriculture)

Context:

You are a Data Scientist working at the USDA. Your department has been tracking the production of various agricultural commodities across different states.

Your datasets include:

`milk_production`, `cheese_production`, `coffee_production`, `honey_production`, `yogurt_production`, and a `state_lookup` table.

The data spans multiple years and states, with varying levels of production for each commodity.

Your manager has requested that you generate insights from this data to aid in future planning and decision-making. You'll need to use SQL queries to answer the questions that come up in meetings, reports, or strategic discussions.

Objectives:

Assess state-by-state production for each commodity.

Identify trends or anomalies.

Offer data-backed suggestions for areas that may need more attention.


NOTE: All answer entries are numeric and only numbers and periods. The autograder does not accept commas for the final project.

# Question 1 Can you find out the total milk production for 2023? Your manager wants this information for the yearly report.
# What is the total milk production for 2023?

In [ ]:
query = """
SELECT
    Year,
    SUM(Value) AS total_milk_production
FROM milk_production
WHERE Year = 2023
GROUP BY Year
"""
result = con.execute(query).df()
pd.set_option('display.float_format', '{:,.0f}'.format)
result



,Year,total_milk_production
0,2023,"91,812,000,000"


# Question 2 Which states had cheese production greater than 100 million in April 2023? The Cheese Department wants to focus their marketing efforts there.
# How many states are there?

In [ ]:
query = """
SELECT
    State_ANSI,
    SUM(Value) AS total_cheese_production
FROM cheese_production
WHERE Year = 2023
  AND Period = 'April'
GROUP BY State_ANSI
HAVING SUM(Value) > 100000000
"""
result = con.execute(query).df()
pd.set_option('display.float_format', '{:,.0f}'.format)
result


,State_ANSI,total_cheese_production


# Question 3 Your manager wants to know how coffee production has changed over the years.
# What is the total value of coffee production for 2011?

In [ ]:
query = """
SELECT
    Year,
    SUM(Value) AS total_coffee_production
FROM coffee_production
WHERE Year = 2011
GROUP BY Year
"""
result = con.execute(query).df()

# 讓數字顯示完整（非科學記號）
pd.set_option('display.float_format', '{:,.0f}'.format)
result


,Year,total_coffee_production
0,2011,"7,600,000"


# Question 4 There's a meeting with the Honey Council next week. Find the average honey production for 2022 so you're prepared.

In [ ]:
query = """
SELECT
    Year,
    AVG(Value) AS avg_honey_production_2022
FROM honey_production
WHERE Year = 2022
GROUP BY Year
"""
result = con.execute(query).df()
pd.set_option('display.float_format', '{:,.0f}'.format)
result


,Year,avg_honey_production_2022
0,2022,"3,133,275"


# Question 5 The State Relations team wants a list of all states names with their corresponding ANSI codes. Can you generate that list?
# What is the State_ANSI code for Florida?

In [ ]:
query = """
SELECT
    State,
    State_ANSI
FROM state_lookup
ORDER BY State

"""
result = con.execute(query).df()
result


,State,State_ANSI
0,ALABAMA,1
1,ALASKA,2
2,ARIZONA,4
3,ARKANSAS,5
4,CALIFORNIA,6
5,COLORADO,8
6,CONNECTICUT,9
7,DELAWARE,10
8,FLORIDA,12
9,GEORGIA,13


# Question 6  For a cross-commodity report, can you list all states with their cheese production values, even if they didn't produce any cheese in April of 2023?
# What is the total for NEW JERSEY?

In [ ]:
query = """
SELECT
    s.State,
    s.State_ANSI,
    COALESCE(SUM(c.Value), 0) AS total_cheese_production
FROM state_lookup s
LEFT JOIN cheese_production c
  ON s.State_ANSI = c.State_ANSI
  AND c.Year = 2023
  AND c.Period = 'April'
GROUP BY s.State, s.State_ANSI
ORDER BY s.State
"""
result = con.execute(query).df()

pd.set_option('display.float_format', '{:,.0f}'.format)
result


,State,State_ANSI,total_cheese_production
0,ALABAMA,1,0
1,ALASKA,2,0
2,ARIZONA,4,0
3,ARKANSAS,5,0
4,CALIFORNIA,6,0
5,COLORADO,8,0
6,CONNECTICUT,9,0
7,DELAWARE,10,0
8,FLORIDA,12,0
9,GEORGIA,13,0


#Question 7 Can you find the total yogurt production for states in the year 2022 which also have cheese production data from 2023? This will help the Dairy Division in their planning.

In [ ]:
query = """

  SELECT SUM(y.Value)
FROM yogurt_production y
WHERE y.Year = 2022 AND y.State_ANSI IN (
    SELECT DISTINCT c.State_ANSI FROM cheese_production c WHERE c.Year = 2023)
"""
result = con.execute(query).df()
pd.set_option('display.float_format', '{:,.0f}'.format)
result


,"sum(y.""Value"")"
0,"1,171,095,000"


# Question 8 List all states from state_lookup that are missing from milk_production in 2023.
# How many states are there?

In [ ]:
query = """
SELECT
    s.State,
    s.State_ANSI
FROM state_lookup s
LEFT JOIN milk_production m
  ON s.State_ANSI = m.State_ANSI
  AND m.Year = 2023
WHERE m.State_ANSI IS NULL
ORDER BY s.State
"""
result = con.execute(query).df()
result


,State,State_ANSI
0,ALABAMA,1
1,ALASKA,2
2,ARKANSAS,5
3,CONNECTICUT,9
4,DELAWARE,10
5,HAWAII,15
6,KENTUCKY,21
7,LOUISIANA,22
8,MAINE,23
9,MARYLAND,24


In [ ]:
con.execute("""
SELECT COUNT(*) AS num_states_missing_milk_2023
FROM state_lookup s
LEFT JOIN milk_production m
  ON s.State_ANSI = m.State_ANSI
  AND m.Year = 2023
WHERE m.State_ANSI IS NULL
""").df()


,num_states_missing_milk_2023
0,26


# Question 9 List all states with their cheese production values, including states that didn't produce any cheese in April 2023. Did Delaware produce any cheese in April 2023?

In [ ]:
query = """
SELECT
    s.State,
    s.State_ANSI,
    COALESCE(SUM(c.Value), 0) AS total_cheese_production_April_2023
FROM state_lookup s
LEFT JOIN cheese_production c
  ON s.State_ANSI = c.State_ANSI
  AND c.Year = 2023
  AND c.Period = 'April'
GROUP BY s.State, s.State_ANSI
ORDER BY s.State
"""
result = con.execute(query).df()

pd.set_option('display.float_format', '{:,.0f}'.format)
result


,State,State_ANSI,total_cheese_production_April_2023
0,ALABAMA,1,0
1,ALASKA,2,0
2,ARIZONA,4,0
3,ARKANSAS,5,0
4,CALIFORNIA,6,0
5,COLORADO,8,0
6,CONNECTICUT,9,0
7,DELAWARE,10,0
8,FLORIDA,12,0
9,GEORGIA,13,0


# Question 10 Find the average coffee production for all years where the honey production exceeded 1 million.

In [ ]:
query = """
SELECT
    AVG(c.Value) AS avg_coffee_production
FROM coffee_production c
WHERE c.Year IN (
    SELECT DISTINCT h.Year
    FROM honey_production h
    GROUP BY h.Year
    HAVING SUM(h.Value) > 1000000
)
"""
result = con.execute(query).df()
pd.set_option('display.float_format', '{:,.0f}'.format)
result


,avg_coffee_production
0,"6,426,667"
